# OSIRIS-HP-003: SPAM Mitigation & Zero-Noise Extrapolation (ZNE) Reproduction
**Target Hardware:** IBM Quantum Heron r2 (`ibm_fez`)
**Job ID:** `dbndsvoou4as73b4cvi0` (46,080 physical shots across 45 native ISA circuits)
**Zenodo Deposition:** [10.5281/zenodo.24031814](https://doi.org/10.5281/zenodo.24031814)
**Governing Protocol:** SYS-021 Zero Self-Supplied Authorization

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Locate experimental data files
analysis_path = Path("mitigated_hp003_analysis.json")
if not analysis_path.exists():
    analysis_path = Path("experiments/hp003/mitigated_hp003_analysis.json")

with open(analysis_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded Benchmark: {data['benchmark_id']}")
print(f"Backend: {data['backend']} | Job ID: {data['job_id']}")
print(f"Readout attenuation factors: Pair 1 = {data['readout_transfer_factors']['pair1_r1_m1']:.5f}, Pair 2 = {data['readout_transfer_factors']['pair2_r2_m2']:.5f}")

## 1. Physical Results Summary Table
Comparison of raw and SPAM-mitigated entanglement fidelities across all 5 arms.

In [ ]:
print(f"{'Arm':<12} | {'CZs':<4} | {'Raw Fidelity':<14} | {'Mitigated Fidelity':<20} | {'Witness W':<10} | {'Z-Score'}")
print("-" * 78)
for arm, stats in data['arm_mitigated_summary'].items():
    cz = stats['cz_count']
    raw_f = stats['raw_mean_fidelity'] * 100.0
    mit_f = stats['mitigated_mean_fidelity'] * 100.0
    sem_f = stats['sem_mitigated_mean'] * 100.0
    w = stats['mitigated_witness']
    z = stats['z_score']
    print(f"{arm:<12} | {cz:<4} | {raw_f:6.2f}%        | {mit_f:6.2f}% +/- {sem_f:4.2f}%    | {w:+.4f}    | {z:6.1f}s")

## 2. Visualization: Raw vs. Mitigated Entanglement Fidelity

In [ ]:
arms = list(data['arm_mitigated_summary'].keys())
raw_vals = [data['arm_mitigated_summary'][a]['raw_mean_fidelity'] * 100.0 for a in arms]
mit_vals = [data['arm_mitigated_summary'][a]['mitigated_mean_fidelity'] * 100.0 for a in arms]
mit_errs = [data['arm_mitigated_summary'][a]['sem_mitigated_mean'] * 100.0 for a in arms]

x = np.arange(len(arms))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
rects1 = ax.bar(x - width/2, raw_vals, width, label='Raw Hardware Telemetry', color='#4575b4', alpha=0.85)
rects2 = ax.bar(x + width/2, mit_vals, width, yerr=mit_errs, capsize=4, label='SPAM-Mitigated Fidelity', color='#d73027', alpha=0.85)

ax.axhline(50.0, color='black', linestyle='--', linewidth=1.2, label='Classical Separability Threshold (50%)')
ax.axhline(data['arm_mitigated_summary']['Arm A']['mitigated_mean_fidelity']*100, color='green', linestyle=':', linewidth=1.5, label='Physical Coherence Ceiling (Arm A Mitigated)')

ax.set_ylabel('Entanglement Fidelity (%)', fontsize=12)
ax.set_title('OSIRIS-HP-003: Multi-Qubit Hayden-Preskill Recovery & Capacity Bound on ibm_fez', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(arms, fontsize=11)
ax.set_ylim(0, 100)
ax.legend(loc='upper right', frameon=True)
ax.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

## 3. Coherence Parity & ZNE Depolarizing Gate Analysis
Evaluating the ratio of Arm B restricted recovery to Arm A positive control ceiling.

In [ ]:
zne = data['zero_noise_extrapolation']
parity = zne['relative_coherence_parity_percent']
eps_a = zne['effective_cz_error_arm_a'] * 100.0
eps_b = zne['effective_cz_error_arm_b'] * 100.0

print(f"Coherence Ceiling Parity (Arm B / Arm A): {parity:.2f}%")
print(f"Effective 2-Qubit CZ Error (Arm A, 48 CZs): {eps_a:.3f}%")
print(f"Effective 2-Qubit CZ Error (Arm B, 49 CZs): {eps_b:.3f}%")
print("\nKey Conclusions:")
for c in zne['conclusions']:
    print(f"  * {c}")